# ETL
Process for creating a combined and cleaned dataset that contains all the information and features the structure necessary to conduct the required A/B tests.

In [1]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

In [2]:
ab_assign = pd.read_parquet("../data/intermediate/ab_assign.parquet")
ab_assign.info()

<class 'pandas.DataFrame'>
RangeIndex: 99910 entries, 0 to 99909
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   customer_id      99910 non-null  str           
 1   group            99910 non-null  str           
 2   assignment_date  99910 non-null  datetime64[us]
dtypes: datetime64[us](1), str(2)
memory usage: 3.5 MB


In [3]:
cust_prof = pd.read_parquet("../data/intermediate/cust_prof.parquet")
cust_prof.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   customer_id               100000 non-null  str           
 1   historical_value_segment  100000 non-null  str           
 2   gender                    100000 non-null  str           
 3   age                       100000 non-null  int64         
 4   region                    100000 non-null  str           
 5   registration_date         100000 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 6.6 MB


In [4]:
transactions = pd.read_parquet("../data/intermediate/transactions.parquet")
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 86498 entries, 90888 to 177385
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     86498 non-null  str           
 1   customer_id  86498 non-null  str           
 2   order_date   86498 non-null  datetime64[us]
 3   revenue      86498 non-null  float64       
 4   items        86498 non-null  int64         
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 4.6 MB


The table that determines the customers is the `ab_assign` table. So we need to enrich with other two tables: `cust_prof` and `transactions`.

In [5]:
# Merge to obtain enriched customer data
assign_cust = pd.merge(ab_assign, cust_prof, on="customer_id", how="left")
assign_cust

,customer_id,group,assignment_date,historical_value_segment,gender,age,region,registration_date
0,C014479,control,2026-01-02,Low,Female,53,West,2023-11-16
1,C045298,control,2026-01-02,Low,Male,34,East,2024-04-19
2,C007346,control,2026-01-02,High,Male,33,West,2021-12-24
3,C015631,control,2026-01-02,Low,Female,37,South,2023-01-19
4,C019112,control,2026-01-02,Medium,Female,25,West,2023-09-05
...,...,...,...,...,...,...,...,...
99905,C068107,control,2026-01-02,Low,Female,25,West,2023-03-04
99906,C014228,test,2026-01-02,Low,Female,61,South,2025-02-21
99907,C048240,control,2026-01-02,Medium,Female,30,East,2024-02-29
99908,C075999,control,2026-01-02,High,Female,41,East,2023-04-15


In [6]:
transactions

,order_id,customer_id,order_date,revenue,items
90888,O00118578,C000066,2026-02-01,55.41,6
90889,O00153672,C000147,2026-02-01,45.14,5
90890,O00061107,C000236,2026-02-01,69.69,6
90891,O00016248,C000263,2026-02-01,178.20,15
90892,O00093898,C000278,2026-02-01,82.46,10
...,...,...,...,...,...
177381,O00162895,C099900,2026-02-28,38.97,4
177382,O00068690,C099919,2026-02-28,153.87,14
177383,O00038944,C099947,2026-02-28,43.59,3
177384,O00021471,C099948,2026-02-28,36.58,3


In [7]:
# Aggregate transactions by customer_id to get summary statistics for each customer
transactions = transactions.groupby("customer_id").agg(
    total_orders = ("order_id", "count"),
    total_revenue = ("revenue", "sum"),
    min_revenue = ("revenue", "min"),
    max_revenue = ("revenue", "max"),
    avg_revenue = ("revenue", "mean"),
    total_items = ("items", "sum"),
    avg_items = ("items", "mean"),
    min_order_date = ("order_date", "min"),
    max_order_date = ("order_date", "max")
).reset_index().copy()

In [8]:
# Merge enriched customer data with transactions to get full dataset
full_data = pd.merge(assign_cust, transactions, on="customer_id", how="left")
full_data

,customer_id,group,assignment_date,historical_value_segment,gender,age,region,registration_date,total_orders,total_revenue,min_revenue,max_revenue,avg_revenue,total_items,avg_items,min_order_date,max_order_date
0,C014479,control,2026-01-02,Low,Female,53,West,2023-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
1,C045298,control,2026-01-02,Low,Male,34,East,2024-04-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
2,C007346,control,2026-01-02,High,Male,33,West,2021-12-24,2.0,115.37,30.75,84.62,57.685,13.0,6.5,2026-02-10,2026-02-15
3,C015631,control,2026-01-02,Low,Female,37,South,2023-01-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
4,C019112,control,2026-01-02,Medium,Female,25,West,2023-09-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99905,C068107,control,2026-01-02,Low,Female,25,West,2023-03-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
99906,C014228,test,2026-01-02,Low,Female,61,South,2025-02-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
99907,C048240,control,2026-01-02,Medium,Female,30,East,2024-02-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
99908,C075999,control,2026-01-02,High,Female,41,East,2023-04-15,1.0,81.30,81.30,81.30,81.300,10.0,10.0,2026-02-17,2026-02-17


In [9]:
# New column to indicate if a customer is active based on total orders
full_data["is_active"] = full_data["total_orders"].apply(lambda x: 1 if x > 0 else 0)
full_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 99910 entries, 0 to 99909
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   customer_id               99910 non-null  str           
 1   group                     99910 non-null  str           
 2   assignment_date           99910 non-null  datetime64[us]
 3   historical_value_segment  99910 non-null  str           
 4   gender                    99910 non-null  str           
 5   age                       99910 non-null  int64         
 6   region                    99910 non-null  str           
 7   registration_date         99910 non-null  datetime64[us]
 8   total_orders              54899 non-null  float64       
 9   total_revenue             54899 non-null  float64       
 10  min_revenue               54899 non-null  float64       
 11  max_revenue               54899 non-null  float64       
 12  avg_revenue               548

In [10]:
# Fill missing values for total items, total revenue, and total orders with 0
full_data[["total_items", "total_revenue", "total_orders"]] = full_data[["total_items", "total_revenue", "total_orders"]].fillna(0)
full_data

,customer_id,group,assignment_date,historical_value_segment,gender,age,region,registration_date,total_orders,total_revenue,min_revenue,max_revenue,avg_revenue,total_items,avg_items,min_order_date,max_order_date,is_active
0,C014479,control,2026-01-02,Low,Female,53,West,2023-11-16,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
1,C045298,control,2026-01-02,Low,Male,34,East,2024-04-19,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
2,C007346,control,2026-01-02,High,Male,33,West,2021-12-24,2.0,115.37,30.75,84.62,57.685,13.0,6.5,2026-02-10,2026-02-15,1
3,C015631,control,2026-01-02,Low,Female,37,South,2023-01-19,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
4,C019112,control,2026-01-02,Medium,Female,25,West,2023-09-05,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99905,C068107,control,2026-01-02,Low,Female,25,West,2023-03-04,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
99906,C014228,test,2026-01-02,Low,Female,61,South,2025-02-21,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
99907,C048240,control,2026-01-02,Medium,Female,30,East,2024-02-29,0.0,0.00,NaN,NaN,NaN,0.0,NaN,NaT,NaT,0
99908,C075999,control,2026-01-02,High,Female,41,East,2023-04-15,1.0,81.30,81.30,81.30,81.300,10.0,10.0,2026-02-17,2026-02-17,1


In [11]:
full_data.to_parquet("../data/processed/full_data.parquet")